**© Copyright AIDENTIFY. All rights reserved.**

# Part 2 | Session 15: Continuous Pretraining (지속 사전학습) with Qwen2.5-1.5B (LoRA)

## 🎯 Continuous Pretraining 개요

**Continuous Pretraining(지속 사전학습)**은 이미 사전 학습된 LLM에 도메인 특화 텍스트를 추가로 학습시키는 방법입니다.

### 왜 Continuous Pretraining이 필요한가?

- 🔹 사전학습 모델은 일반적인 지식은 풍부하지만, 특정 도메인 지식이 부족할 수 있습니다
- 🔹 의료, 법률, 금융 등 전문 분야의 용어와 지식을 모델에 주입할 수 있습니다
- 🔹 Instruction Tuning과 달리, **비구조화된 텍스트**(plain text)를 그대로 학습합니다
- 🔹 Next Token Prediction 방식으로 도메인 언어 패턴을 학습합니다

### 학습 목표

- ✅ Continuous Pretraining의 개념과 활용 사례 이해
- ✅ 도메인 텍스트 데이터 전처리 (토큰화 + 청크 분할)
- ✅ LoRA를 활용한 효율적 Continuous Pretraining 수행
- ✅ 학습 전후 도메인 지식 변화 비교

### 실습 환경

| 항목 | 설정 |
|------|------|
| 모델 | Qwen2.5-1.5B (4bit 양자화) |
| 방법 | LoRA (r=16) |
| GPU | RTX 4060 (8GB VRAM) |
| 데이터 | 도메인 텍스트 (AI/ML 관련) |

## 1️⃣ 환경 설정 및 모델 로드 (Qwen2.5-1.5B, 4bit)

In [ ]:
# 필수 라이브러리 임포트
import torch
import gc
import os
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import warnings
warnings.filterwarnings('ignore')

print("✅ 라이브러리 임포트 완료")
print(f"PyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# GPU 메모리 모니터링 함수
def print_gpu_memory(tag=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**3
        total = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"[{tag}] GPU: {allocated:.1f}GB / {total:.1f}GB")

print_gpu_memory("시작")

In [ ]:
# 모델 설정
MODEL_NAME = "Qwen/Qwen2.5-1.5B"  # Instruct 버전이 아닌 base 모델 사용
OUTPUT_DIR = "./output/continuous_pretraining"

# 4bit 양자화 설정 (RTX 4060 메모리 절약)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

print("✅ 양자화 설정 완료")
print(f"📌 모델: {MODEL_NAME}")
print(f"📌 양자화: 4bit NF4")

In [ ]:
# 토크나이저 로드
print("⏳ 토크나이저 로딩 중...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# pad_token 설정
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"✅ 토크나이저 로드 완료")
print(f"📌 Vocab 크기: {tokenizer.vocab_size:,}")
print(f"📌 EOS 토큰: {tokenizer.eos_token}")
print(f"📌 PAD 토큰: {tokenizer.pad_token}")

In [ ]:
# 모델 로드 (4bit 양자화)
print("⏳ 모델 로딩 중... (4bit 양자화)")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# gradient checkpointing 활성화 (메모리 절약)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()

print("✅ 모델 로드 완료")
print(f"📌 모델 파라미터 수: {model.num_parameters():,}")
print_gpu_memory("모델 로드 후")

## 2️⃣ 도메인 텍스트 데이터 준비

Continuous Pretraining에서는 **비구조화된 텍스트**(plain text)를 사용합니다.

- 📄 Instruction-Output 쌍이 아닌 일반 텍스트
- 📄 도메인 특화 문서, 논문, 매뉴얼 등
- 📄 모델이 Next Token Prediction으로 도메인 언어를 학습

In [ ]:
# 도메인 텍스트 데이터 로드
data_path = "../data/samples/domain_text_sample.txt"

print("⏳ 도메인 텍스트 데이터 로딩 중...")
with open(data_path, "r", encoding="utf-8") as f:
    domain_text = f.read()

print(f"✅ 데이터 로드 완료")
print(f"📌 전체 텍스트 길이: {len(domain_text):,} 글자")
print(f"📌 줄 수: {len(domain_text.splitlines())}줄")
print(f"\n--- 데이터 미리보기 (처음 500자) ---")
print(domain_text[:500])

In [ ]:
# 텍스트를 문단 단위로 분할
paragraphs = [p.strip() for p in domain_text.split("\n\n") if p.strip()]

print(f"✅ 문단 분할 완료")
print(f"📌 총 문단 수: {len(paragraphs)}")
print(f"\n--- 문단별 길이 ---")
for i, p in enumerate(paragraphs[:5]):
    print(f"  문단 {i+1}: {len(p)}자 - {p[:60]}...")

## 3️⃣ 텍스트 데이터를 토큰화 및 청크 분할

Continuous Pretraining에서는 텍스트를 일정 길이의 **청크(chunk)**로 분할합니다.

- 🔹 max_seq_length 이내로 텍스트를 자릅니다
- 🔹 각 청크가 하나의 학습 샘플이 됩니다
- 🔹 Next Token Prediction: input_ids와 labels가 동일합니다

In [ ]:
MAX_SEQ_LENGTH = 1024  # RTX 4060 안전 설정

# 전체 텍스트를 하나로 합치고 토큰화
full_text = "\n\n".join(paragraphs)
print(f"⏳ 텍스트 토큰화 중...")

tokenized = tokenizer(
    full_text,
    return_tensors=None,
    add_special_tokens=False
)

all_input_ids = tokenized["input_ids"]
print(f"✅ 토큰화 완료")
print(f"📌 전체 토큰 수: {len(all_input_ids):,}")
print(f"📌 max_seq_length: {MAX_SEQ_LENGTH}")

In [ ]:
# 토큰을 max_seq_length 크기의 청크로 분할
chunks = []
for i in range(0, len(all_input_ids), MAX_SEQ_LENGTH):
    chunk = all_input_ids[i:i + MAX_SEQ_LENGTH]
    if len(chunk) >= 64:  # 너무 짧은 청크는 제외
        chunks.append(chunk)

print(f"✅ 청크 분할 완료")
print(f"📌 총 청크 수: {len(chunks)}")
print(f"📌 각 청크 길이: {[len(c) for c in chunks]}")

In [ ]:
# HuggingFace Dataset 형태로 변환
dataset_dict = {
    "input_ids": chunks,
    "attention_mask": [[1] * len(c) for c in chunks],
    "labels": chunks  # Causal LM: labels = input_ids
}

dataset = Dataset.from_dict(dataset_dict)

print(f"✅ Dataset 생성 완료")
print(f"📌 학습 샘플 수: {len(dataset)}")
print(f"📌 Dataset 컬럼: {dataset.column_names}")
print(f"\n--- 첫 번째 샘플 디코딩 (처음 200자) ---")
print(tokenizer.decode(dataset[0]["input_ids"][:100]))

## 4️⃣ LoRA 설정 및 적용

LoRA를 사용하여 메모리 효율적으로 Continuous Pretraining을 수행합니다.

| 파라미터 | 값 | 설명 |
|----------|-----|------|
| r | 16 | LoRA 랭크 |
| lora_alpha | 32 | 스케일링 팩터 |
| target_modules | q_proj, k_proj, v_proj, o_proj | 어텐션 레이어 |
| task_type | CAUSAL_LM | Next Token Prediction |

In [ ]:
# LoRA 설정
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print("✅ LoRA 설정 완료")
print(f"📌 r (랭크): {lora_config.r}")
print(f"📌 alpha: {lora_config.lora_alpha}")
print(f"📌 target_modules: {lora_config.target_modules}")
print(f"📌 dropout: {lora_config.lora_dropout}")

In [ ]:
# LoRA 적용
print("⏳ LoRA 어댑터 적용 중...")
model = get_peft_model(model, lora_config)

# 학습 가능한 파라미터 확인
model.print_trainable_parameters()

print_gpu_memory("LoRA 적용 후")

## 5️⃣ 학습 실행 (Trainer API)

RTX 4060에 안전한 학습 설정:

- 🔹 `per_device_train_batch_size=1` - 메모리 절약
- 🔹 `gradient_accumulation_steps=8` - 실효 배치 크기 = 8
- 🔹 `fp16=True` - 혼합 정밀도 학습
- 🔹 `max_steps` - 적은 데이터이므로 에포크 대신 스텝 기반

In [ ]:
# 학습 전 모델 응답 저장 (비교용)
print("="*60)
print("📋 학습 전 모델 응답 (Before Training)")
print("="*60)

test_prompts = [
    "LoRA(Low-Rank Adaptation)는",
    "트랜스포머(Transformer) 아키텍처는",
    "QLoRA는 모델을",
    "RAG(Retrieval-Augmented Generation)는",
    "Continuous Pretraining은"
]

model.eval()

# 추론을 위해 모든 파라미터를 bfloat16으로 통일
for name, param in model.named_parameters():
    if param.dtype == torch.float32:
        param.data = param.data.to(torch.bfloat16)
before_responses = []

for prompt in test_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            temperature=1.0
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    before_responses.append(response)
    print(f"\n🔹 프롬프트: {prompt}")
    print(f"   응답: {response[:200]}")

model.train()
print("\n" + "="*60)

In [ ]:
# 학습 인자 설정
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,        # RTX 4060 안전 설정
    gradient_accumulation_steps=8,         # 실효 배치 크기 = 8
    learning_rate=2e-4,
    bf16=True,                             # 혼합 정밀도
    logging_steps=1,
    save_strategy="epoch",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=False,
    gradient_checkpointing=True,
)

print("✅ TrainingArguments 설정 완료")
print(f"📌 batch_size: {training_args.per_device_train_batch_size}")
print(f"📌 gradient_accumulation: {training_args.gradient_accumulation_steps}")
print(f"📌 effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"📌 epochs: {training_args.num_train_epochs}")
print(f"📌 learning_rate: {training_args.learning_rate}")
print(f"📌 fp16: {training_args.fp16}")

In [ ]:
# Data Collator (Causal LM용 - MLM=False)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Causal LM이므로 MLM은 False
)

# Trainer 초기화
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

print("✅ Trainer 초기화 완료")
print_gpu_memory("학습 시작 전")

In [ ]:
# 학습 실행
print("🚀 Continuous Pretraining 학습 시작!")
print("="*60)

train_result = trainer.train()

print("="*60)
print("✅ 학습 완료!")
print(f"📌 Total steps: {train_result.global_step}")
print(f"📌 Training loss: {train_result.training_loss:.4f}")
print_gpu_memory("학습 완료 후")

## 6️⃣ 학습 전후 비교 (도메인 지식 질문)

동일한 프롬프트로 학습 전후의 응답을 비교합니다.

- 🔹 도메인 용어에 대한 이해도 변화 확인
- 🔹 텍스트 생성 품질 비교
- 🔹 도메인 지식이 잘 주입되었는지 평가

In [ ]:
# 학습 후 모델 응답 생성
print("="*60)
print("📋 학습 전후 비교 (Before vs After)")
print("="*60)

model.eval()

# 추론을 위해 모든 파라미터를 bfloat16으로 통일
for name, param in model.named_parameters():
    if param.dtype == torch.float32:
        param.data = param.data.to(torch.bfloat16)
after_responses = []

for i, prompt in enumerate(test_prompts):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            temperature=1.0
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    after_responses.append(response)
    
    print(f"\n{'='*60}")
    print(f"🔹 프롬프트: {prompt}")
    print(f"\n  [Before] {before_responses[i][:200]}")
    print(f"\n  [After]  {response[:200]}")

print(f"\n{'='*60}")
print("📌 도메인 텍스트로 학습한 후, 관련 용어에 대한 연속 생성이 더 자연스러워졌는지 확인하세요.")

## 7️⃣ 모델 저장

학습된 LoRA 어댑터만 저장합니다. (원본 모델은 저장하지 않음)

In [ ]:
# LoRA 어댑터 저장
save_path = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"✅ 모델 저장 완료: {save_path}")

# 저장된 파일 크기 확인
total_size = 0
for root, dirs, files in os.walk(save_path):
    for f in files:
        fp = os.path.join(root, f)
        size = os.path.getsize(fp)
        total_size += size
        print(f"  📄 {f}: {size/1024/1024:.1f} MB")

print(f"\n📌 전체 어댑터 크기: {total_size/1024/1024:.1f} MB")
print(f"📌 원본 모델 크기 대비 매우 작은 크기입니다!")

In [ ]:
# GPU 메모리 정리
del model, trainer
gc.collect()
torch.cuda.empty_cache()
print_gpu_memory("메모리 정리 후")
print("✅ GPU 메모리 정리 완료")

## 📝 정리 및 핵심 요약

### 이번 실습에서 배운 내용

| 항목 | 내용 |
|------|------|
| Continuous Pretraining | 도메인 텍스트를 Next Token Prediction으로 추가 학습 |
| 데이터 준비 | 텍스트 → 토큰화 → 고정 길이 청크 분할 |
| LoRA 적용 | 4bit 양자화 + LoRA로 메모리 효율적 학습 |
| 학습 결과 | 도메인 용어와 개념에 대한 생성 품질 향상 |

### 핵심 포인트

- 🎯 **Continuous Pretraining은 비구조화 텍스트**를 사용합니다 (instruction 형식 아님)
- 🎯 **Next Token Prediction** 방식: labels = input_ids
- 🎯 **도메인 지식 주입**에 효과적 (의료, 법률, 금융 등)
- 🎯 **LoRA를 사용하면** 8GB VRAM으로도 수행 가능
- 🎯 실제 활용 시에는 **더 많은 도메인 데이터**가 필요합니다

### 다음 단계

- ➡️ **Session 16**: Instruction Tuning - 지시-응답 형식 데이터로 파인튜닝
- ➡️ Continuous Pretraining → Instruction Tuning 순서로 학습하면 더 효과적!